# Pakistani Name → Gender Classifier
Trains a model to predict gender from a name alone. Labels aren't looked up from a pre-labeled file — they're decoded directly from the CNIC in this notebook, so every step is visible: CNIC parity gives gender, the CNIC's first digit gives province.

**Before running:** upload `ATL_IT_merged_clean.csv` (the NTN + NAME file, before any labeling) to this Colab session (folder icon on the left → upload), or mount Google Drive if you stored it there.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import joblib

## 1. Load raw registry data

In [ ]:
# Option A: manual upload
# from google.colab import files
# uploaded = files.upload()

# Option B: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# path = '/content/drive/MyDrive/ATL_IT_merged_clean.csv'

path = 'ATL_IT_merged_clean.csv'  # adjust if using Drive
df = pd.read_csv(path, dtype=str)
print(df.shape)
df.head()

## 2. Decode gender and province from the CNIC
For individuals, Pakistan's FBR uses the CNIC number directly as the NTN, so the `NTN` column here doubles as the CNIC for most rows.

**Gender** — NADRA's own convention: the last digit of a 13-digit CNIC is odd for male, even for female. This is a hard rule, not a model, so it's exact for every row that has a valid CNIC.

**Province** — the first digit of the CNIC is commonly documented (NADRA/Wikipedia and multiple independent sources) as a province code:

| Digit | Province/territory |
|---|---|
| 1 | Khyber Pakhtunkhwa |
| 2 | FATA (now merged into KP) |
| 3 | Punjab |
| 4 | Sindh |
| 5 | Balochistan |
| 6 | Islamabad Capital Territory |
| 7 | Gilgit-Baltistan |

**A real caveat worth showing your instructor:** running this rule on the actual dataset gives an implausible result — digit `4` (Sindh) accounts for only ~0.03% of rows, when Sindh (home to Karachi) should be a large share of registered taxpayers. The most likely explanation is that this NTN column isn't a true CNIC for every row — some individuals, especially older registrations concentrated in commercial hubs like Karachi, kept a legacy NTN issued before NADRA's 2011 policy of using the CNIC as the NTN, so the first-digit province rule doesn't apply to them. Digits `8` and `9` also show up in real counts even though they aren't part of the standard 1–7 scheme, so this notebook labels them `Unknown` rather than guessing. Treat the PROVINCE column as a best-effort decode, not verified ground truth.

In [ ]:
# Keep only rows with a real 13-digit CNIC/NTN — the digit rules below only mean something for those
df = df[df['NTN'].astype(str).str.len() == 13].copy()

last_digit = df['NTN'].str[-1].astype(int)
df['GENDER'] = last_digit.apply(lambda d: 'Male' if d % 2 == 1 else 'Female')

PROVINCE_MAP = {
    '1': 'Khyber Pakhtunkhwa',
    '2': 'FATA',
    '3': 'Punjab',
    '4': 'Sindh',
    '5': 'Balochistan',
    '6': 'Islamabad',
    '7': 'Gilgit-Baltistan',
}
first_digit = df['NTN'].str[0]
df['PROVINCE'] = first_digit.map(PROVINCE_MAP).fillna('Unknown')

print(df['GENDER'].value_counts())
print()
print(df['PROVINCE'].value_counts())

## 3. Clean names
Uppercase + strip whitespace so casing differences ("Ali Raza" vs "ALI RAZA") don't become separate features.

In [ ]:
df['NAME'] = df['NAME'].astype(str).str.upper().str.strip()
df = df[df['NAME'].str.len() > 0].reset_index(drop=True)
print(df['GENDER'].value_counts())
print(df['GENDER'].value_counts(normalize=True))

## 4. Train / test split
Stratified so both splits keep the same Male/Female ratio as the full (imbalanced) dataset.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['NAME'], df['GENDER'],
    test_size=0.2,
    random_state=42,
    stratify=df['GENDER']
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Build the model
Character n-gram TF-IDF is the standard approach for name→gender classification — it picks up on prefixes/suffixes (e.g. names ending in "A", "EEN", "ISH") without needing a name dictionary.

`class_weight='balanced'` corrects for the 81/19 Male/Female imbalance so the model doesn't just learn to always predict Male.

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=3)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1))
])

model.fit(X_train, y_train)

## 6. Evaluate
Look at precision/recall/F1 per class, not just accuracy — accuracy alone is misleading on imbalanced data (a model that always guesses "Male" would still score ~81%).

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=['Male', 'Female'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Male', 'Female'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 7. Try it on new names

In [ ]:
def predict_gender(name):
    name = name.upper().strip()
    pred = model.predict([name])[0]
    proba = model.predict_proba([name])[0]
    classes = model.classes_
    prob_dict = dict(zip(classes, proba))
    return pred, prob_dict

for test_name in ['Ayesha Khan', 'Muhammad Bilal', 'Zainab Fatima', 'Ali Hassan']:
    pred, probs = predict_gender(test_name)
    print(f'{test_name:20s} -> {pred:8s}  {probs}')

## 8. Save the model
Download this so you don't have to retrain every session.

In [ ]:
joblib.dump(model, 'name_gender_classifier.joblib')

# from google.colab import files
# files.download('name_gender_classifier.joblib')

---
### Notes / things to tune next
- **Imbalance:** if recall on Female names is still weak after this baseline, try oversampling the minority class (e.g. `imblearn`'s `RandomOverSampler`) in addition to `class_weight='balanced'`.
- **Multi-word names:** many entries have 3+ words (e.g. "MUHAMMAD JAMAL NASIR ALI"). The char n-gram model uses the whole string as-is; if accuracy is disappointing, try extracting just the *first* token as a separate feature, since that's usually the most gender-indicative part in Pakistani naming conventions.
- **Speed:** with 2.4M rows this should train in well under a minute on Colab's default CPU runtime, since it's a sparse linear model, not a deep net.
- **Scaling up later:** if you want higher accuracy than this baseline gives, a character-level LSTM/GRU (via Keras) or fine-tuning a small transformer on character sequences would be the next step up — but get this baseline's numbers first before deciding whether that's worth the extra complexity.
- **PROVINCE column:** it's carried through the whole notebook (e.g. `df.groupby('PROVINCE')['GENDER'].value_counts()`) if your instructor wants a province breakdown too, but given the Sindh anomaly noted in step 2, treat any province-based conclusion as provisional.